<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  <b>02. 🔐 Secure Password Manager</b>
</div>


### 📌 Project Overview
In this project, we design a console-based Password Manager that securely handles user credentials.
Using Python's built-in `sqlite3` engine for database storage, alongside key derivation standard hashing processes, users can safely persist their logins without exposing raw credentials.

#### 🔑 Key Concepts Covered:
- SQLite persistent database schema construction, updates, and indexing
- Master Password validation via salt-based PBKDF2 (`hashlib.pbkdf2_hmac`) with 100,000 iterations
- Basic key derivation: generating a decryption key dynamically without storing the master password on disk
- Clean CLI prompt design with secure user inputs


In [ ]:
import os
import sqlite3
import hashlib
import base64

class PasswordManager:
    def __init__(self, db_name='passwords.db'):
        self.db_name = db_name
        self.conn = sqlite3.connect(self.db_name)
        self.create_tables()
        
    def create_tables(self):
        cursor = self.conn.cursor()
        # Create master table to store master hash and salt
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS master (
                id INTEGER PRIMARY KEY,
                hash TEXT NOT NULL,
                salt TEXT NOT NULL
            )
        ''')
        # Create credentials table
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS credentials (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                site TEXT NOT NULL UNIQUE,
                username TEXT NOT NULL,
                encrypted_password TEXT NOT NULL
            )
        ''')
        self.conn.commit()
        
    def derive_key(self, master_password, salt):
        """Derive a 256-bit symmetric key from password and salt."""
        return hashlib.pbkdf2_hmac(
            'sha256',
            master_password.encode(),
            salt.encode(),
            100000
        )
        
    def simple_encrypt(self, plaintext, key):
        """Symmetric XOR encryption using the derived key bytes."""
        encrypted = bytearray()
        for i, char in enumerate(plaintext):
            key_byte = key[i % len(key)]
            encrypted.append(ord(char) ^ key_byte)
        return base64.b64encode(encrypted).decode()
        
    def simple_decrypt(self, ciphertext, key):
        """Symmetric XOR decryption using the derived key bytes."""
        try:
            decoded = base64.b64decode(ciphertext.encode())
            decrypted = []
            for i, byte in enumerate(decoded):
                key_byte = key[i % len(key)]
                decrypted.append(chr(byte ^ key_byte))
            return ''.join(decrypted)
        except Exception:
            return None
            
    def initialize_master(self, master_password):
        cursor = self.conn.cursor()
        cursor.execute('SELECT * FROM master WHERE id = 1')
        if cursor.fetchone():
            return False  # Already configured
            
        salt = os.urandom(16).hex()
        key = self.derive_key(master_password, salt)
        verification_hash = hashlib.sha256(key).hexdigest()
        
        cursor.execute('INSERT INTO master (id, hash, salt) VALUES (1, ?, ?)', (verification_hash, salt))
        self.conn.commit()
        return True
        
    def verify_master(self, master_password):
        cursor = self.conn.cursor()
        cursor.execute('SELECT hash, salt FROM master WHERE id = 1')
        row = cursor.fetchone()
        if not row:
            return None
            
        saved_hash, salt = row
        key = self.derive_key(master_password, salt)
        input_hash = hashlib.sha256(key).hexdigest()
        
        if input_hash == saved_hash:
            return key  # Return key to use for encryption/decryption
        return None
        
    def add_credential(self, site, username, password, key):
        cursor = self.conn.cursor()
        encrypted_pwd = self.simple_encrypt(password, key)
        try:
            cursor.execute(
                'INSERT INTO credentials (site, username, encrypted_password) VALUES (?, ?, ?)',
                (site.lower(), username, encrypted_pwd)
            )
            self.conn.commit()
            return True
        except sqlite3.IntegrityError:
            return False
            
    def get_credential(self, site, key):
        cursor = self.conn.cursor()
        cursor.execute('SELECT username, encrypted_password FROM credentials WHERE site = ?', (site.lower(),))
        row = cursor.fetchone()
        if not row:
            return None
        username, encrypted_pwd = row
        decrypted_pwd = self.simple_decrypt(encrypted_pwd, key)
        return username, decrypted_pwd
        
    def list_sites(self):
        cursor = self.conn.cursor()
        cursor.execute('SELECT site, username FROM credentials')
        return cursor.fetchall()
        
    def close(self):
        self.conn.close()


In [ ]:
# Demonstration of PasswordManager lifecycle in Python
db_filename = 'passwords_demo.db'
if os.path.exists(db_filename):
    os.remove(db_filename)  # Start clean
    
pm = PasswordManager(db_filename)
master_pw = 'FaizySecure2026!'

# 1. Initialize DB
if pm.initialize_master(master_pw):
    print('🔑 Master password set successfully.')
else:
    print('⚠️ Master password already initialized.')

# 2. Authenticate
user_key = pm.verify_master(master_pw)
if user_key:
    print('✅ Authentication successful!')
    
    # 3. Add credentials
    pm.add_credential('github.com', 'mohd-faizy', 'G1tHubPassword@@123', user_key)
    pm.add_credential('linkedin.com', 'faizy_linked', 'Link3dInP@ss!', user_key)
    print('📝 Logins added for github.com and linkedin.com.')
    
    # 4. List registered portals
    print('\nRegistered portals:')
    for site, user in pm.list_sites():
        print(f'- Portal: {site} | Username: {user}')
        
    # 5. Retrieve decrypted value
    site_to_query = 'github.com'
    result = pm.get_credential(site_to_query, user_key)
    if result:
        usr, pwd = result
        print(f'\n🔓 Decrypted account for {site_to_query}:')
        print(f'   Username: {usr}')
        print(f'   Password: {pwd}')
else:
    print('❌ Authentication failed.')

pm.close()
